[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/10_gqa_solution.ipynb)

# 🔴 Solution: Grouped Query Attention

Reference solution for GQA — MHA with shared KV heads.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

In [7]:
class GroupQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.d_head = d_model // num_heads
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_head)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_head)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, S, d_model = x.shape
        Q = self.W_q(x).view(B, S, self.num_heads, self.d_head).transpose(-2, -3)
        K = self.W_k(x).view(B, S, self.num_kv_heads, self.d_head ).transpose(-2, -3)
        V = self.W_v(x).view(B, S, self.num_kv_heads, self.d_head ).transpose(-2, -3)
        repeat = self.num_heads // self.num_kv_heads
        K = K.repeat_interleave(repeat, dim=-3)
        V = V.repeat_interleave(repeat, dim=-3)
        scale = self.d_head ** 0.5
        scores = torch.einsum('...qd,...kd->...qk', Q, K) / scale
        weight = torch.softmax(scores, dim=-1)
        atten = torch.einsum('...qk,...kd->...qd', weight, V)
        output = atten.transpose(-2, -3).reshape(B, S, self.d_model)
        return self.W_o(output)


In [2]:
from timeit import repeat


class GroupQueryAttention():
    def __init__(self, d_model: int, h_n: int, kv_n: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.h_n = h_n
        self.kv_n = kv_n
        self.d_head = d_model // h_n
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, kv_n * self.d_head)
        self.W_v = nn.Linear(d_model, kv_n * self.d_head)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        *B, S, d_model = x.shape
        Q = self.W_q(x).view(*B, S, self.h_n, self.d_head).transpose(-2, -3)
        K = self.W_k(x).view(*B, S, self.kv_n, self.d_head).transpose(-2, -3)
        V = self.W_v(x).view(*B, S, self.kv_n, self.d_head).transpose(-2, -3)
        repeat = self.h_n // self.kv_n
        K = K.repeat_interleave(repeat, dim=-3)
        V = V.repeat_interleave(repeat, dim=-3)
        scores = torch.einsum('...qd,...kd->...qk', Q, K) / self.d_head ** 0.5
        weight = torch.softmax(scores, dim=-1)
        atten = torch.einsum('...qk,...kd->...qd', weight, V)
        output = atten.transpose(-2, -3).reshape(*B, S, d_model)
        return self.W_o(output)




In [3]:
gqa = GroupQueryAttention(32, 8, 2)
print('Output:', gqa.forward(torch.randn(1, 4, 32)).shape)

Output: torch.Size([1, 4, 32])


In [ ]:
from torch_judge import check
check('gqa')